# Feed Amount Analysis (RF-DETR) - Log-Normal Kalman Filter

This notebook estimates feed consumption rate from per-frame RF-DETR bounding box areas.

**Theory:** RF-DETR detection noise is multiplicative (proportional to box size).
Taking `log(area)` converts it to additive Gaussian noise for optimal Kalman filtering.

Key features:
- 2D state vector: `[log_area, consumption_rate]`
- Unbiased area recovery: `exp(μ + P₁₁/2)`
- 1-sigma confidence interval from filter covariance
- True one-step-ahead prediction via constant-velocity model
- **Model:** RF-DETR (detects: debris, feed, fish, lumpfish)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import os
import logging

# Suppress all warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*use_return_dict.*')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

# Suppress transformers logging before import
for logger_name in ['transformers', 'transformers.modeling_utils']:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

# Set default font
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. Kalman Filter Implementation

In [ ]:
class LogNormalKalmanFilter:
    """
    2D Log-Normal Kalman Filter for feed area estimation.

    State vector: x = [mu, mu_dot]
      mu      = log(true feed area)
      mu_dot  = d(log A)/dt  [1/frame]

    Zero-detection handling: prediction-only (no measurement update).
    Mathematically correct: missing detection ≠ area=0.
    Outlier rejection: Mahalanobis distance gate rejects false positives.
    """

    def __init__(self, sigma_R: float = 0.5, sigma_Q: float = 0.3, dt: float = 1.0,
                 outlier_threshold: float = 5.0):
        self.dt = dt
        self.sigma_R = sigma_R
        self.outlier_threshold = outlier_threshold

        self.x = np.zeros(2)
        self.P = np.eye(2)
        self.F = np.array([[1.0, dt], [0.0, 1.0]])
        self.h = np.array([1.0, 0.0])
        self.Q = sigma_Q ** 2 * np.array([
            [dt ** 4 / 4, dt ** 3 / 2],
            [dt ** 3 / 2, dt ** 2],
        ])
        self.R_scalar = sigma_R ** 2

        self.initialized = False
        self.consecutive_zeros = 0
        self.outliers_rejected = 0

    def update(self, area_measurement: float) -> float:
        if not self.initialized:
            if area_measurement <= 0:
                return 0.0
            self.x = np.array([np.log(area_measurement), 0.0])
            self.P = np.diag([self.R_scalar, 1.0])
            self.initialized = True
            return self.area_estimate

        # Prediction step (always runs)
        x_pred = self.F @ self.x
        P_pred = self.F @ self.P @ self.F.T + self.Q

        if area_measurement <= 0:
            # No detection: prediction only, let uncertainty grow naturally
            # DO NOT push the estimate toward zero — missing detection ≠ zero area
            self.consecutive_zeros += 1
            self.x = x_pred
            self.P = P_pred
        else:
            self.consecutive_zeros = 0
            y = np.log(area_measurement)
            innovation = y - self.h @ x_pred

            # Outlier gate: reject detections far from prediction (false positives)
            S = float(self.h @ P_pred @ self.h) + self.R_scalar
            mahal_sq = (innovation ** 2) / S
            if mahal_sq > self.outlier_threshold ** 2:
                self.outliers_rejected += 1
                self.x = x_pred
                self.P = P_pred
            else:
                K = (P_pred @ self.h) / S
                self.x = x_pred + K * innovation
                self.P = (np.eye(2) - np.outer(K, self.h)) @ P_pred

        return self.area_estimate

    @property
    def area_estimate(self) -> float:
        # Unbiased log-normal mean: E[A] = exp(mu + sigma²/2)
        return float(np.exp(np.clip(self.x[0] + self.P[0, 0] / 2, -10, 20)))

    @property
    def consumption_rate(self) -> float:
        return float(self.x[1])

    @property
    def area_confidence_interval(self):
        log_std = np.sqrt(self.P[0, 0])
        lower = float(np.exp(np.clip(self.x[0] - log_std, -10, 20)))
        upper = float(np.exp(np.clip(self.x[0] + log_std, -10, 20)))
        return lower, upper

## 2. Configuration Parameters

In [ ]:
CONFIG = {
    'video_path': '../data/fish_feeding.mp4',
    'model_path': '../models/rf-detr-fish-regular.pth',
    'confidence_threshold': 0.3,
    'device': 'cuda',
    'sigma_R': 0.5,              # Log-space measurement noise std (0.5 ≈ ±65% relative error)
    'sigma_Q': 0.3,              # Process noise std — higher = filter follows signal faster
    'outlier_threshold': 5.0,    # Mahalanobis gate (sigma): reject detections > 5σ from prediction
    'max_frames': None,
}

print('Configuration parameters:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 3. Load RF-DETR Model and Video

In [ ]:
from rfdetr.detr import RFDETR

print(f'加載模型: {CONFIG["model_path"]}')
model = RFDETR.from_checkpoint(CONFIG['model_path'])
model.device = CONFIG['device']

class_names = {0: 'debris', 1: 'feed', 2: 'fish', 3: 'lumpfish'}
print(f'類別: {class_names}')

video_path = CONFIG['video_path']
print(f'\\n加載視頻: {video_path}')
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f'無法打開視頻: {video_path}')

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'視頻信息:')
print(f'  總幀數: {total_frames}')
print(f'  FPS: {fps}')
print(f'  解像度: {width}x{height}')
print(f'  總時長: {total_frames/fps:.2f}秒')

## 4. Run Inference and Calculate Feed Area (RF-DETR Only)

In [ ]:
frame_numbers    = []
feed_areas_raw   = []
feed_areas_filtered = []
consumption_rates   = []
predictions      = []
conf_upper       = []
conf_lower       = []

kf = LogNormalKalmanFilter(
    sigma_R=CONFIG['sigma_R'],
    sigma_Q=CONFIG['sigma_Q'],
    outlier_threshold=CONFIG['outlier_threshold'],
    dt=1.0,
)

frame_idx  = 0
max_frames = CONFIG['max_frames'] or total_frames

print(f'開始推論，共 {min(max_frames, total_frames)} 幀...')
print(f'信心度閾值: {CONFIG["confidence_threshold"]}')
print(f'sigma_R={CONFIG["sigma_R"]},  sigma_Q={CONFIG["sigma_Q"]},  outlier_threshold={CONFIG["outlier_threshold"]}σ')
print()

In [ ]:
import sys

frames_with_detections = []
frames_with_feed = []
feed_detections_per_frame = []
feed_confidence_per_frame = []
feed_sizes_per_frame = []

while frame_idx < max_frames:
    ret, frame = cap.read()
    if not ret:
        print(f'\n到達視頻末尾在 Frame {frame_idx}')
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    detections = model.predict(rgb_frame, threshold=CONFIG['confidence_threshold'])

    frame_feed_area = 0.0
    frame_feed_count = 0
    frame_feed_confs = []
    frame_feed_sizes = []

    if detections is not None and len(detections) > 0:
        frames_with_detections.append(frame_idx)
        if frame_idx < 5:
            print(f'\nFrame {frame_idx}: 检测到 {len(detections)} 个对象')
            for box, cls_id, conf in zip(detections.xyxy, detections.class_id, detections.confidence):
                print(f'  Class {cls_id} ({class_names[cls_id]}): conf={conf:.3f}')

        for box, cls_id, conf in zip(detections.xyxy, detections.class_id, detections.confidence):
            if cls_id == 1:  # feed class
                x1, y1, x2, y2 = box
                area = (x2 - x1) * (y2 - y1)
                frame_feed_area += area
                frame_feed_count += 1
                frame_feed_confs.append(conf)
                frame_feed_sizes.append(area)

    if frame_feed_count > 0:
        frames_with_feed.append(frame_idx)

    feed_detections_per_frame.append(frame_feed_count)
    feed_confidence_per_frame.append(frame_feed_confs)
    feed_sizes_per_frame.append(frame_feed_sizes)

    frame_numbers.append(frame_idx)
    feed_areas_raw.append(frame_feed_area)

    filtered = kf.update(frame_feed_area)
    feed_areas_filtered.append(filtered)
    consumption_rates.append(kf.consumption_rate)
    predictions.append(kf.predict_next() if hasattr(kf, 'predict_next') else filtered)

    lo, hi = kf.area_confidence_interval
    conf_lower.append(lo)
    conf_upper.append(hi)

    if (frame_idx + 1) % 100 == 0:
        sys.stdout.write(f'\r已處理: {frame_idx + 1}/{min(max_frames, total_frames)} 幀')
        sys.stdout.flush()

    frame_idx += 1

cap.release()
print(f'\n\n推論完成！共處理 {len(frame_numbers)} 幀')

print(f'\n【RF-DETR 檢測診斷】')
print(f'有檢測結果的幀數: {len(frames_with_detections)}/{len(frame_numbers)} ({len(frames_with_detections)/len(frame_numbers)*100:.1f}%)')
print(f'檢測到 feed 的幀數: {len(frames_with_feed)}/{len(frame_numbers)} ({len(frames_with_feed)/len(frame_numbers)*100:.1f}%)')
print(f'平均每幀 feed 檢測數: {np.mean(feed_detections_per_frame):.2f}')
print(f'最多一幀 feed 檢測數: {np.max(feed_detections_per_frame)}')

all_feed_confs = [c for confs in feed_confidence_per_frame for c in confs]
if all_feed_confs:
    print(f'\nFeed 檢測信心度統計:')
    print(f'  最小值: {np.min(all_feed_confs):.3f}')
    print(f'  最大值: {np.max(all_feed_confs):.3f}')
    print(f'  平均值: {np.mean(all_feed_confs):.3f}')
    print(f'  標準差: {np.std(all_feed_confs):.3f}')

print(f'\n【Kalman Filter 診斷】')
print(f'  異常值拒絕幀數: {kf.outliers_rejected} / {len(frames_with_feed)} ({kf.outliers_rejected/max(len(frames_with_feed),1)*100:.1f}%)')

## 5. Data Statistics

In [ ]:
feed_areas_raw      = np.array(feed_areas_raw)
feed_areas_filtered = np.array(feed_areas_filtered)
consumption_rates   = np.array(consumption_rates)
predictions         = np.array(predictions)
conf_lower          = np.array(conf_lower)
conf_upper          = np.array(conf_upper)
frame_numbers       = np.array(frame_numbers)

print('=' * 65)
print('飼料面積統計 (RF-DETR)')
print('=' * 65)

print('\\n原始值 (RF-DETR 測量):')
print(f'  最小值: {feed_areas_raw.min():,.0f} pixels²')
print(f'  最大值: {feed_areas_raw.max():,.0f} pixels²')
print(f'  平均值: {feed_areas_raw.mean():,.0f} pixels²')
print(f'  標準差: {feed_areas_raw.std():,.0f} pixels²')

print('\\nLog-Normal KF 濾波後 (無偏估計):')
print(f'  最小值: {feed_areas_filtered.min():,.0f} pixels²')
print(f'  最大值: {feed_areas_filtered.max():,.0f} pixels²')
print(f'  平均值: {feed_areas_filtered.mean():,.0f} pixels²')
print(f'  標準差: {feed_areas_filtered.std():,.0f} pixels²')

noise_reduction = (1 - feed_areas_filtered.std() / feed_areas_raw.std()) * 100 if feed_areas_raw.std() > 0 else 0
print(f'  降噪效果: {noise_reduction:.1f}%')

print('\\n飼料消耗速率 d(log A)/dt  [1/frame]:')
print(f'  均值:   {consumption_rates.mean():.5f}')
print(f'  標準差: {consumption_rates.std():.5f}')
print(f'  最小值: {consumption_rates.min():.5f}  (最快消耗)')
print(f'  最大值: {consumption_rates.max():.5f}  (最快增加)')

pct_per_frame = consumption_rates.mean() * 100
pct_per_sec   = pct_per_frame * fps
print(f'\\n  平均 {pct_per_frame:.3f}% / frame  →  {pct_per_sec:.2f}% / sec')

## 6. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(frame_numbers, feed_areas_raw,
        label='Raw (RF-DETR)', color='red', alpha=0.4, linewidth=1.0)
ax.fill_between(frame_numbers, conf_lower, conf_upper,
                alpha=0.25, color='steelblue', label='1σ Confidence Band')
ax.plot(frame_numbers, feed_areas_filtered,
        label='Log-Normal KF (unbiased)', color='steelblue', linewidth=2.5)

ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
ax.set_ylabel('Feed Area (pixels²)', fontsize=12, fontweight='bold')
ax.set_title('Feed Area — Log-Normal Kalman Filter vs Raw RF-DETR', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}k'))

plt.tight_layout()
plt.savefig('feed_amount_analysis_rfdetr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feed_amount_analysis_rfdetr.png')

## 7. Export Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'Frame':                frame_numbers,
    'Raw_Feed_Area':        feed_areas_raw,
    'Filtered_Feed_Area':   feed_areas_filtered,
    'Consumption_Rate':     consumption_rates,
    'Consumption_Rate_Pct': consumption_rates * 100,
    'Predicted_Feed_Area':  predictions,
    'Conf_Lower_1sigma':    conf_lower,
    'Conf_Upper_1sigma':    conf_upper,
    'Filtering_Error':      np.abs(feed_areas_raw - feed_areas_filtered),
})

csv_path = 'feed_amount_analysis_rfdetr_results.csv'
results_df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

print('\\nData preview (first 10 rows):')
print(results_df.head(10).to_string())

print('\\nSummary:')
print(results_df.describe())

## 8. Final Report

In [ ]:
print('\\n' + '=' * 70)
print(' ' * 10 + '飼料量分析 — RF-DETR + Log-Normal KF 完整報告')
print('=' * 70)

print(f'\\n【基本信息】')
print(f'  視頻文件:     {CONFIG["video_path"]}')
print(f'  RF-DETR 模型: {CONFIG["model_path"]}')
print(f'  處理幀數:     {len(frame_numbers)}')
print(f'  視頻時長:     {len(frame_numbers)/fps:.2f} 秒')
print(f'  信心度閾值:   {CONFIG["confidence_threshold"]}')

print(f'\\n【模型參數】')
print(f'  sigma_R: {CONFIG["sigma_R"]} (測量噪聲)')
print(f'  sigma_Q: {CONFIG["sigma_Q"]} (過程噪聲)')

print(f'\\n【飼料面積統計】')
print(f'  原始值:   {feed_areas_raw.min():,.0f} ~ {feed_areas_raw.max():,.0f} pixels²')
print(f'  濾波後:   {feed_areas_filtered.min():,.0f} ~ {feed_areas_filtered.max():,.0f} pixels²')
print(f'  降噪效果: {noise_reduction:.1f}%')

print(f'\\n【消耗速率】')
print(f'  均值: {consumption_rates.mean()*100:.4f}% / frame')
print(f'  標準差: {consumption_rates.std()*100:.4f}% / frame')

print(f'\\n【輸出文件】')
print(f'  ✓ feed_amount_analysis_rfdetr.png')
print(f'  ✓ feed_amount_analysis_rfdetr_results.csv')

print('\\n' + '=' * 70)
print('分析完成！')
print('=' * 70)